# Axis III, RQ11 -- Prompt-Injection Bias Probe: When Confused by Injected/Conflicting Text, Does the Model Default Left or Right? only -- 50 subjects

A trimmed copy of `RQ1_to_RQ11_axis3_experiments_kaggle.ipynb` that implements **only
RQ11 -- Prompt-Injection Bias Probe: When Confused by Injected/Conflicting Text, Does the Model Default Left or Right?** (previously capped at 5 subjects). Everything else (setup, model loading,
helpers) is identical to the full notebook. This notebook runs **50 subjects in one
pass** -- no batching.

Six prompt variants injecting conflicting or irrelevant text alongside/instead of a clean "left X"/"right X" instruction -- see the methodology doc's RQ11 section for the full writeup.


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Setup working directories (Kaggle environment)

In [ ]:
import os
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
for _rq in ('rq1', 'rq2', 'rq3', 'rq4', 'rq5', 'rq6', 'rq7', 'rq8', 'rq9', 'rq10', 'rq11'):
    (RESULTS_ROOT / _rq).mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT, 'in subfolders rq1/ ... rq11/')


## 3. Install VoxTell

In [ ]:
import subprocess, sys, re
from pathlib import Path

# Installs voxtell + deps, a pinned transformers, and a clean torch+torchvision pair BEFORE
# torch is ever imported in this kernel, then requires a kernel restart. `pip install voxtell`
# pins torch<2.9 and can silently downgrade Kaggle's preinstalled torch; separately, Kaggle's
# preinstalled torch build has been observed shipping self-inconsistent (breaks on
# `from torch._dynamo import OptimizedModule`, which VoxTell's predictor uses). Both need a
# clean, paired reinstall done via `pip show` (never `import torch`) before the restart.
_marker = Path('/kaggle/working/.env_setup_done')
if _marker.exists():
    print('environment already installed/fixed earlier in this session -- skipping. '
          f'({_marker.read_text()})')
else:
    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

    print('installing voxtell + deps...')
    _pip('voxtell', 'nibabel', 'pandas', 'matplotlib', 'huggingface_hub')

    _show = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                            capture_output=True, text=True, check=True).stdout
    _torch_version = re.search(r'^Version: (.+)$', _show, re.MULTILINE).group(1).split('+')[0]
    # torch, torchvision, transformers, and accelerate are ALL installed in ONE pip command --
    # a prior version of this cell installed torch+torchvision first, then transformers+accelerate
    # in a SEPARATE pip call. That second call had no reason to touch torch, but pip's resolver
    # silently upgraded it anyway (observed: torch 2.11.0 -> 2.14.1) to satisfy some dependency of
    # accelerate/transformers, re-breaking the torch/torchvision pairing the first call had just
    # fixed, and transformers 4.57.6 then failed its own internal feature probing against that
    # newer, untested torch -- surfacing as "ImportError: cannot import name 'GenerationMixin'
    # from transformers.generation", not as an obvious torch-version error. Pinning torch with
    # `==` inside the SAME resolver pass as transformers/accelerate stops pip from silently
    # reconsidering it afterward.
    print(f'reinstalling torch=={_torch_version} + torchvision + transformers==4.57.6 + accelerate '
          f'together in one resolver pass (this can take a few minutes)...')
    _pip('--force-reinstall', '--no-cache-dir', f'torch=={_torch_version}', 'torchvision',
         'transformers==4.57.6', 'accelerate>=1.0.0')

    _marker.write_text(f'torch=={_torch_version}')

    if 'torch' in sys.modules:
        # Something earlier in THIS kernel session already imported torch -- the reinstall just
        # changed what's on disk, not what's already loaded in memory, so a manual kernel restart
        # really is required before continuing. On a fresh "Save & Run All" this never fires (this
        # cell runs before anything else imports torch -- the only earlier cell is `!nvidia-smi`,
        # a shell command, not a Python import) -- it only fires if you interactively re-ran this
        # cell after already running a later cell that imports torch.
        raise RuntimeError(
            'torch was already imported earlier in this session, before the reinstall -- '
            'RESTART THE KERNEL NOW, then re-run this notebook from the top (this cell will '
            'then just print "skipping"). This should not happen on a fresh Save & Run All run.'
        )
    print('torch/torchvision/transformers reinstalled cleanly -- no kernel restart needed '
          '(nothing in this kernel session has imported torch yet).')


## 4. Clone the research repo

Uses a Kaggle Secret named `GITHUB_PAT` (Add-ons -> Secrets) -- never hardcode a token here
(an earlier sibling notebook did, and it had to be flagged and rotated).

In [ ]:
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'teamwork/neurite-oasis-dataset-rq-analysis', '--depth', '1', _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    raise RuntimeError(f'git clone failed (exit {_result.returncode}):\n{_stderr_scrubbed}')

RESEARCH_REPO = Path(_dest)
print('cloned into', RESEARCH_REPO)


## 4b. Download the 100-subject Neurite-OASIS dataset from Hugging Face

Uses a Kaggle Secret named `HF_TOKEN` -- never hardcode a token here.

In [ ]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Add a Kaggle Secret named HF_TOKEN before running this cell."

NEURITE_OASIS_DIR = Path(snapshot_download(
    repo_id='Fatema142/neurite-oasis-100-rq-analysis',
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir=str(WORKING_DIR / 'neurite_oasis_100'),
))
print('downloaded to', NEURITE_OASIS_DIR)


## 5. Download and load the VoxTell model

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import voxtell.inference.predictor as _vp

class _NoQwen:  # VoxTellPredictor.__init__ would load a full fp32 Qwen3-4B (~16 GB) into host RAM; cell 14
    @staticmethod  # then loads the fp16 one we actually use. Holding both at once can kill the Kaggle kernel.
    def from_pretrained(*a, **k):
        return torch.nn.Identity()

_orig_automodel = _vp.AutoModel
_vp.AutoModel = _NoQwen
try:
    predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
finally:
    _vp.AutoModel = _orig_automodel
# VoxTellPredictor.__init__ loads the checkpoint on CPU and only moves the network to
# `device` inside predict_sliding_window_return_logits() -- every mechanistic helper below
# calls model.encoder/model.decoder directly (bypassing that method), so the network must be
# moved here or every such call hits 'Input type ... and weight type ... should be the same'.
predictor.network = predictor.network.to(device)
model = predictor.network
print('VoxTell predictor ready on', device, '| num_heads (RQ1 fusion channels):', model.num_heads)
assert model.num_heads == 32, f'RQ1 assumes 32 fusion channels, checkpoint has num_heads={model.num_heads}'


## 5b. Fix: text encoder OOM on a single T4/P100 (encoder on CPU + embedding cache)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

# The 4B-param fp16 text encoder (~8 GB) is kept on the CPU, NOT on the GPU: on a 15 GB T4 it left too
# little room for the segmentation network (RQ3's run_interventions died with "out of GPU memory even
# with one output per pass"). Embeddings are cached per prompt string, so each distinct prompt is
# computed once (briefly moving the encoder to the GPU) and every later call is a dictionary lookup --
# no 8 GB CPU<->GPU transfer per call like the original VoxTellPredictor.embed_text_prompts.
predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()
if getattr(predictor, 'tokenizer', None) is None:
    predictor.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')

import threading
from voxtell.utils.text_embedding import last_token_pool, wrap_with_instruction

_EMBED_CACHE = {}  # prompt string -> (D,) float32 CPU tensor
_embed_cache_lock = threading.Lock()


@torch.inference_mode()
def _embed_missing(prompts):
    """Embed `prompts` (list of str) with the encoder moved onto the GPU just for this call."""
    dev = predictor.device
    torch.cuda.empty_cache()
    predictor.text_backbone.to(dev)
    try:
        wrapped = wrap_with_instruction(list(prompts))
        tok = predictor.tokenizer(wrapped, padding=True, truncation=True,
                                  max_length=predictor.max_text_length, return_tensors='pt')
        tok = {k: v.to(dev) for k, v in tok.items()}
        out = predictor.text_backbone(**tok)
        emb = last_token_pool(out.last_hidden_state, tok['attention_mask']).float().cpu()
    finally:
        predictor.text_backbone.to('cpu')
        torch.cuda.empty_cache()
    return emb


def _embed_text_prompts_fp16_safe(text_prompts):
    """Same output as VoxTellPredictor.embed_text_prompts: (1, n_prompts, D) float32 on predictor.device."""
    if isinstance(text_prompts, str):
        text_prompts = [text_prompts]
    text_prompts = list(text_prompts)
    with _embed_cache_lock:
        missing = list(dict.fromkeys(t for t in text_prompts if t not in _EMBED_CACHE))
        if missing:
            for t, e in zip(missing, _embed_missing(missing)):
                _EMBED_CACHE[t] = e
        emb = torch.stack([_EMBED_CACHE[t] for t in text_prompts])
    return emb.view(1, len(text_prompts), -1).to(predictor.device)


predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder: fp16 on CPU + per-prompt embedding cache; tokenizer loaded:',
      predictor.tokenizer is not None)


## 6. Helpers (I/O, manifest, RAS axis bookkeeping -- same logic as `scripts/common3d.py`)

In [ ]:
import csv
from pathlib import Path

import numpy as np
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader so predicted masks and ground truth end up
    in the exact same RAS-reoriented array space. Returns (1, X, Y, Z)."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props

def spatial_view(arr_4d):
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

# NibabelIOWithReorient returns axes (Superior-Inferior, Anterior-Posterior, Right-Left) --
# verified empirically against real predictions (left/right kidney centroids differ ~85 voxels
# on this axis, ~3 on the others, for every subject).
AXCODES = ("S", "A", "R")
RL_AXIS = AXCODES.index("R")

def load_manifest_rot000(person, dataset_name):
    """Loads the 100-subject manifest from the Hugging Face-downloaded dataset dir
    (NEURITE_OASIS_DIR, set in cell 4b) -- this dataset has no rotation sweep, so there's
    no rotation_angle_deg column to filter on, unlike the sibling-dataset notebooks this
    helper was adapted from."""
    path = NEURITE_OASIS_DIR / f'manifest_{dataset_name}_100.csv'
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def iou_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(inter / union)

print('helpers ready')


## 7. Load the experiment code

Imports `scripts/voxtell_mechanistic.py` (exports `encode_mask_rle`/`decode_mask_rle`/
`encode_array_b64`/`decode_array_b64`, the shared CSV-persistence helpers used by every RQ
below) and `scripts/voxtell_experiments.py` (`run_interventions(..., return_masks=True)`,
reused directly by RQ3/RQ4) from the cloned repo, via `sys.path`.

In [ ]:
import sys

sys.path.insert(0, str(RESEARCH_REPO / 'scripts'))
import voxtell_mechanistic as vm
import voxtell_experiments as ve

print('loaded voxtell_mechanistic from', vm.__file__)
print('loaded voxtell_experiments from', ve.__file__)
assert hasattr(vm, 'encode_mask_rle'), 'need the CSV-persistence helpers in voxtell_mechanistic.py'
assert 'return_masks' in ve.run_interventions.__code__.co_varnames, 'need run_interventions(return_masks=True)'


## 8. Subjects and organs -- ALL 100 staged subjects x every bilateral organ pair (scope for all 5 RQs)

`SUBJECTS` is discovered from the manifest itself (every distinct `subject_id` staged by
`scripts/stage_neurite_oasis_100.py` into `Fatema142/neurite-oasis-100-rq-analysis` -- all 100
subjects, not a hand-picked subset), rather than hardcoded, so this always matches whatever is
actually in the downloaded dataset. `ORGAN_CORES = None` keeps every bilateral organ pair
present in the manifest -- the full 16-pair OASIS-1 list in `RQ1_Fusion_Channel_Analysis.md`
(cerebral white matter, cerebral cortex, lateral ventricle, inferior lateral ventricle,
cerebellum white matter, cerebellum cortex, thalamus, caudate, putamen, pallidum, hippocampus,
amygdala, accumbens, ventral DC, vessel, choroid plexus). Midline structures (3rd/4th
ventricle, brain stem) have no left/right counterpart and are dropped automatically -- they
carry no laterality signal for any of the 5 RQs.

**Running all 5 RQs over all 100 subjects in one sitting is a lot of GPU time** (RQ1 alone is
~16 organ pairs x 2 forward passes x 100 subjects). If a session times out, lower
`MAX_SUBJECTS` below to a smaller number first, confirm the full notebook runs end to end, then
remove the cap and let it run the full 100 (e.g. across several Kaggle sessions, appending to
the same CSVs with `mode='a'` if you restart partway through).

In [ ]:
DATASET_NAME = 'neurite_oasis'
PERSON = 'faria'
ORGAN_CORES = None  # None = every bilateral organ pair present in the manifest
MAX_SUBJECTS = None  # None = all 100 staged subjects; set to e.g. 10 for a quick dry run

_all_rows = load_manifest_rot000(PERSON, DATASET_NAME)
_all_subject_ids = sorted({r['subject_id'] for r in _all_rows})
assert len(_all_subject_ids) == 100, f'expected 100 staged subjects, manifest has {len(_all_subject_ids)}'
SUBJECTS = [(PERSON, sid) for sid in (_all_subject_ids if MAX_SUBJECTS is None else _all_subject_ids[:MAX_SUBJECTS])]
print(f'using {len(SUBJECTS)} of {len(_all_subject_ids)} staged subjects')

subject_organs = {}
for person, sid in SUBJECTS:
    rows = [r for r in _all_rows if r['subject_id'] == sid]  # _all_rows already loaded above, once
    rows = [r for r in rows if r['organ'].split()[0].lower() in ('left', 'right')]
    if ORGAN_CORES:
        rows = [r for r in rows if r['organ'].split(None, 1)[1] in ORGAN_CORES]
    have = {r['organ'] for r in rows}
    rows = [r for r in rows if ve.mirror(r['organ']) in have]          # keep complete left/right pairs only
    assert rows, f'no bilateral organs for {person}/{sid}'
    subject_organs[sid] = {'person': person, 'rows': rows}

n_pairs = {sid: len({r['organ'].split(None, 1)[1] for r in v['rows']}) for sid, v in subject_organs.items()}
print(len(subject_organs), 'subjects; bilateral organ pairs per subject (first 5 shown):',
      dict(list(n_pairs.items())[:5]))


def load_subject(sid):
    info = subject_organs[sid]
    rows = info['rows']
    base = NEURITE_OASIS_DIR
    img_arr, _ = load_volume_ras(base / rows[0]['image_path'])
    label_vol, _ = load_seg_ras(base / rows[0]['label_path'])
    organs = [r['organ'] for r in rows]
    ids = {r['organ']: int(r['label_id']) for r in rows}
    prompts = {r['organ']: r['prompt_text'] for r in rows}
    return info['person'], img_arr, label_vol, organs, ids, prompts


## 8b. Resumable, crash-safe CSV writer + dual-GPU dispatch

Every RQ loop below writes each subject's rows to CSV **immediately** (not once at the end), and skips subjects already present in its output CSV on startup -- so a Kaggle session timeout/restart/crash loses at most the subject(s) in flight, and rerunning a cell picks up where it left off instead of starting over. If 2 GPUs are visible (Kaggle accelerator "GPU T4 x2"), subjects are split across both and processed concurrently via a second full predictor+model loaded onto the second device.

In [ ]:
import csv
import os
import queue
import threading
import time
import traceback
from pathlib import Path

# Use every visible GPU, not just predictor.device -- set the Kaggle accelerator to "GPU T4 x2"
# (Settings -> Accelerator) to actually get a second device here; with a single-GPU accelerator
# this safely falls back to running everything on predictor.device as before.
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICES = [torch.device(f'cuda:{i}') for i in range(N_GPUS)] if N_GPUS else [predictor.device]
# Safe default: ONE GPU. Building a second VoxTell predictor on cuda:1 is the heaviest step of this cell
# (host RAM + GPU memory) and is what has been killing the Kaggle kernel. Set True only once everything
# runs cleanly on one GPU and you want the extra speed.
USE_SECOND_GPU = False
if not USE_SECOND_GPU:
    DEVICES = [predictor.device]
print('visible CUDA devices:', DEVICES)

PREDICTORS_BY_DEVICE = {predictor.device: predictor}
MODELS_BY_DEVICE = {predictor.device: model}


_embed_lock = threading.Lock()


def _build_predictor_on(device):
    """A second, fully independent VoxTellPredictor (segmentation network + its own
    Qwen3-Embedding-4B text backbone) on `device`, so the RQ loops below can process different
    subjects on different GPUs at the same time instead of everything funneling through the one
    predictor on cuda:0. Uses the same fp16 text backbone and on-device-resident embed_text_prompts
    as cell 14, just bound to this device instead."""
    from voxtell.inference.predictor import VoxTellPredictor
    import voxtell.inference.predictor as _vp

    class _NoQwen:  # the constructor would otherwise load a full fp32 Qwen3-4B (~16 GB) into host RAM,
        @staticmethod  # which can kill the Kaggle kernel; we never use that copy anyway (see below)
        def from_pretrained(*a, **k):
            return torch.nn.Identity()

    _orig_automodel = _vp.AutoModel
    _vp.AutoModel = _NoQwen
    try:
        p = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
    finally:
        _vp.AutoModel = _orig_automodel
    p.network = p.network.to(device)
    # NO second Qwen3 copy on this GPU: a 4B fp16 text encoder is ~8 GB, and on a 15 GB T4 that
    # left no room for the segmentation network's activations (CUDA OOM on every subject). The text
    # encoder is cheap, so all devices share the ONE on the main predictor (serialized by a lock)
    # and the resulting embedding is moved onto this device. The CPU copy VoxTellPredictor's
    # constructor loads is dropped too (frees ~16 GB of host RAM).
    p.text_backbone = None
    main = PREDICTORS_BY_DEVICE[predictor.device]

    def _embed(text_prompts):
        with _embed_lock:
            return main.embed_text_prompts(text_prompts).to(device)

    p.embed_text_prompts = _embed
    return p


import gc
gc.collect()  # drop tensors/graphs still referenced by earlier failed runs (e.g. via stored tracebacks)
for _device in DEVICES:
    if _device.type == 'cuda':
        with torch.cuda.device(_device):
            torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info(_device)
        print(f'{_device}: {_free / 2**30:.1f} / {_total / 2**30:.1f} GiB free before loading')
        if _device not in PREDICTORS_BY_DEVICE and _free < 8 * 2**30:
            raise RuntimeError(f'{_device} only has {_free / 2**30:.1f} GiB free -- something from an earlier '
                               'run is still holding GPU memory. Restart the kernel (Run -> Restart) and re-run '
                               'the setup cells; re-running this cell cannot free it.')

for _device in DEVICES:
    if _device not in PREDICTORS_BY_DEVICE:
        print('loading a second VoxTell + Qwen3 text encoder on', _device, '...')
        PREDICTORS_BY_DEVICE[_device] = _build_predictor_on(_device)
    MODELS_BY_DEVICE[_device] = PREDICTORS_BY_DEVICE[_device].network

WORKER_DEVICES = list(PREDICTORS_BY_DEVICE.keys())
print(f'ready: {len(WORKER_DEVICES)} worker device(s) ->', WORKER_DEVICES)

_csv_lock = threading.Lock()


def completed_ids(csv_path, id_field='subject_id'):
    """Subject IDs already present in csv_path -- lets a rerun after a crash, timeout, or kernel
    restart skip everything that's already on disk instead of starting the whole RQ over."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return set()
    with open(csv_path, newline='') as f:
        return {row[id_field] for row in csv.DictReader(f)}


def append_rows(csv_path, fieldnames, rows):
    """Appends rows to csv_path immediately and fsyncs to disk, writing the header only the first
    time. Called once per subject (not once per whole RQ), so a crash mid-run loses at most the
    subject(s) currently in flight, never the entire run's results."""
    if not rows:
        return
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with _csv_lock:
        write_header = not csv_path.exists()
        with open(csv_path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if write_header:
                w.writeheader()
            w.writerows(rows)
            f.flush()
            os.fsync(f.fileno())


def run_over_subjects(sids, per_subject_fn, desc=''):
    """Runs per_subject_fn(predictor_d, model_d, device, sid) for every sid in sids, with one
    worker thread per entry in WORKER_DEVICES pulling from a shared queue -- different subjects
    run on different GPUs concurrently. per_subject_fn must call append_rows() itself for every
    CSV it produces, as soon as that subject's rows are ready. Always prints this RQ's total wall
    time (and, for an actual run, the time/subject) so you can see where the budget is going."""
    sids = list(sids)
    t_start = time.time()
    if not sids:
        print(f'[{desc}] nothing to do -- every requested subject is already in the CSV(s) (0s)')
        return
    q = queue.Queue()
    for sid in sids:
        q.put(sid)
    remaining = len(sids)
    progress_lock = threading.Lock()

    def worker(device):
        nonlocal remaining
        predictor_d, model_d = PREDICTORS_BY_DEVICE[device], MODELS_BY_DEVICE[device]
        while True:
            try:
                sid = q.get_nowait()
            except queue.Empty:
                return
            t0 = time.time()
            try:
                per_subject_fn(predictor_d, model_d, device, sid)
                with progress_lock:
                    remaining -= 1
                    left = remaining
                print(f'[{desc}][{device}] {sid}: done in {time.time() - t0:.0f}s ({left} left)')
            except Exception:
                print(f'[{desc}][{device}] {sid}: FAILED')
                traceback.print_exc()
            finally:
                q.task_done()

    threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in WORKER_DEVICES]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    elapsed = time.time() - t_start
    print(f'[{desc}] all {len(sids)} requested subjects processed in {elapsed:.0f}s '
          f'({elapsed / len(sids):.1f}s/subject, {len(WORKER_DEVICES)} GPU(s))')


## 9. Subjects for this run -- the first 50

`SUBJECTS_SUBSET` is the first 50 subjects out of every subject staged in the
manifest (loaded above as `subject_organs`).


In [ ]:
N_SUBJECTS = 50  # <-- change this one number to use more/fewer subjects

_all_loaded_sids = list(subject_organs)
assert N_SUBJECTS <= len(_all_loaded_sids), (
    f'requested {N_SUBJECTS} subjects, but only {len(_all_loaded_sids)} are loaded')

SUBJECTS_SUBSET = _all_loaded_sids[:N_SUBJECTS]
print(f'running the first {len(SUBJECTS_SUBSET)} of {len(_all_loaded_sids)} loaded subjects '
      f'-> {SUBJECTS_SUBSET}')


---
# RQ11 -- Prompt-Injection Bias Probe: When Confused by Injected/Conflicting Text, Does the Model Default Left or Right?

Six prompt variants injecting conflicting or irrelevant text alongside/instead of a clean "left X"/"right X" instruction -- see the methodology doc's RQ11 section for the full writeup.

Unchanged from the full notebook except it runs over `SUBJECTS_SUBSET` (50 subjects,
defined above) instead of a hardcoded 5-subject cap.


In [ ]:
rq11_csv = RESULTS_ROOT / 'rq11' / 'rq11_prompt_injection_bias.csv'
rq11_fields = ['dataset', 'person', 'subject_id', 'organ', 'variant', 'prompt', 'n_pred_voxels',
               'dice_left_organ', 'dice_right_organ', 'call', 'dice_vs_clean_left',
               'dice_vs_clean_right', 'volume_shape', 'gt_left_mask_rle', 'gt_right_mask_rle',
               'clean_left_mask_rle', 'clean_right_mask_rle', 'pred_mask_rle']


def _rq11_one_subject(predictor_d, model_d, device, sid):
    rows = []
    person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
    volume_shape = 'x'.join(map(str, label_vol.shape))
    left_organs = sorted(o for o in organs if o.startswith('left') and ve.mirror(o) in organs)
    for organ in left_organs:
        res, gt_left_rle, gt_right_rle, clean_left_rle, clean_right_rle = ve.run_prompt_injection_bias(
            predictor_d, img_arr, label_vol, organ, ids, prompts, return_masks=True,
            variants=[v for v in ve.INJECTION_VARIANTS if not v.startswith('jailbreak_style')])
        for r in res:
            rows.append({'dataset': DATASET_NAME, 'person': person, 'subject_id': sid,
                          'volume_shape': volume_shape, 'gt_left_mask_rle': gt_left_rle,
                          'gt_right_mask_rle': gt_right_rle, 'clean_left_mask_rle': clean_left_rle,
                          'clean_right_mask_rle': clean_right_rle, **r})

    append_rows(rq11_csv, rq11_fields, rows)


_rq11_done = completed_ids(rq11_csv)
_rq11_todo = [sid for sid in SUBJECTS_SUBSET if sid not in _rq11_done]
print(f'RQ11: {len(_rq11_done)} subjects already in {rq11_csv.name}, {len(_rq11_todo)} left to run')
run_over_subjects(_rq11_todo, _rq11_one_subject, desc='rq11')


---
## Outputs -- RQ11 -- Prompt-Injection Bias Probe: When Confused by Injected/Conflicting Text, Does the Model Default Left or Right? CSVs for 50 subjects

Masks are in the **full volume** coordinate frame (`volume_shape` column, varies per
subject) -- decode with `vm.decode_mask_rle(cell, volume_shape_tuple)`, where
`volume_shape_tuple = tuple(int(x) for x in row['volume_shape'].split('x'))`.

| RQ | File | Columns worth knowing | Decode masks with |
|---|---|---|---|
| RQ11 | `rq11_prompt_injection_bias.csv` | `variant`, `prompt`, `call` (which side it resolved to), `dice_vs_clean_left`/`dice_vs_clean_right`, `volume_shape`, GT masks + clean-baseline masks + `pred_mask_rle` | `vm.decode_mask_rle(cell, volume_shape)` |

After downloading, concatenate these CSVs with any earlier runs' CSVs (matching filenames) to
build up a larger combined result set -- rows are keyed by `subject_id`, so a plain
`pandas.concat` + `drop_duplicates(subset=['subject_id', ...])` is enough.
